# Thumbnail Derivatives

Open the gallery page on the laptop in your study and everything is instant. Open the same gallery on the laptop in the kitchen over the family Wi-Fi and watch several hundred megabytes of original-resolution JPEGs stream past — every photo at 8 MP, every photo loaded into a 256×256 thumbnail slot. The browse flow from notebook 12 returns `presigned_url` pointing at the S3 **original** because there is nothing else to point at. The Flet `Image` control downscales for display after the bytes have already crossed the network.

[The cost is paid in bandwidth]{.mark}. A library of 50 000 photos browsed one page of 30 at a time downloads `30 × ~5 MB = 150 MB` per page if the user scrolls to the bottom of a year. Worse, a person album with 500 photos — each photo containing a detected face — runs `generate_face_thumbnail` 500 times, which in notebook 12's implementation each time downloads the original from S3 (`12-photo-app.ipynb:[16]`). That is 500 × 5 MB = 2.5 GB of reads to render one person's album cover thumbnails. Multiply by every person in your library.

This notebook builds the **derivative tier**: three JPEG derivatives per photo (`small` 256 px, `medium` 512 px, `large` 1024 px) generated at index time, stored in a sibling S3 bucket (or prefix), and served from the same presigned-URL flow as the original. Browse serves small; album view serves medium; the detail page serves large (and the original only when explicitly requested). Face crops read the medium derivative, not the original. The half-million-bytes-of-wasted-bandwidth problem becomes a 25-KB-thumbnail problem.

<br>

**Design constraints.**
1. **One image → three derivatives** generated once at index time, not on demand. On-demand generation amortizes work but adds tail latency and forces every API request to babysit an image decoder.
2. **The original is preserved.** We never overwrite the original; derivatives live in a separate key prefix (`thumbnails/{sha}/{small|medium|large}.jpg`).
3. **Backfill is cheap.** Pre-existing rows without derivatives (the bucket-synced legacy photos from notebook 12) should be picked up by the same pipeline that generates them for new uploads.
4. **Hash-stable keys.** A derivative's key is derived from the photo's sha256, so regenerating the same derivative produces the same key and an idempotent `PUT`. This is what PHT:05's idempotent re-index relies on.

---

## Derivative Sizes and Why Three

Two sizes is the minimum (list + detail). Three sizes matches what every photo app depuis Apple Photos ships. The breakpoints are not arbitrary — they map to common viewport widths:

| Size | Long-edge px | Typical use | JPEG quality | Approx bytes (4:3 photo) |
|---|---|---|---|---|
| `small` | 256 | Timeline grid, search results, person album thumbnails | 70 | 15–25 KB |
| `medium` | 512 | Person album detail view, year-scope scroll | 80 | 40–80 KB |
| `large` | 1024 | Detail page, before requesting the original | 85 | 150–300 KB |
| `original` | full | "View at full resolution" explicit action only | as-saved | 2–10 MB |

:::{.callout-note}
The `small` long-edge is **256**, not 128 or 200. At 128 px a 4:3 photo renders at 128 × 96, which is visibly blurry on a 200 DPI laptop. At 256 px the same photo renders sharply on a 2× retina screen at the typical 128 × 96 layout slot, because the browser/Flet scales down. 256 is the long-edge that matches "looks crisp at 2× DPI."

:::


## Derivative Generation

The generation function takes a photo row and an S3 client, downloads the original, generates three derivatives with Pillow's `LANCZOS` resampling (the right resampler both down and up), and `PUT`s each derivative to its derivative bucket. The function is a pure transform — no global state, no DB writes — which makes it trivially testable and idempotent. Calling it twice on the same photo with the same source bytes puts the same three keys on S3.


In [ ]:
import io
from dataclasses import dataclass

try:
    from PIL import Image as PILImage
    _PIL_AVAILABLE = True
except ImportError:
    _PIL_AVAILABLE = False


DERIVATIVE_SIZES: dict[str, tuple[int, int]] = {
    "small":  (256,  70),     # long-edge, JPEG quality
    "medium": (512,  80),
    "large":  (1024, 85),
}

DERIVATIVE_BUCKET = "my-photo-app-thumbnails"


@dataclass
class PhotoRow:
    photo_id:  str
    s3_key:    str
    sha256:    str


def _derivative_key(photo_sha: str, size_name: str) -> str:
    """Hash-stable key: regenerating produces the same key.

    Key shape:
        thumbnails/{sha_prefix}/{sha}/{size}.jpg
    The sha_prefix is the first 2 hex chars — same sharding trick as PHT:01's
    original key, which keeps ListObjectsV2 prefixes balanced.
    """
    return f"thumbnails/{photo_sha[:2]}/{photo_sha}/{size_name}.jpg"


def _resize_lanczos(img_bytes: bytes, long_edge: int, quality: int) -> bytes:
    """Resize so the long edge equals `long_edge`, keep aspect ratio, LANCZOS.
    Falls back to the input bytes when Pillow cannot decode them (e.g. demos
    or test fixtures that inject synthetic bytes) so the surrounding pipeline
    remains robust."""
    if not _PIL_AVAILABLE:
        return img_bytes          # fallback: return original (demo only)
    try:
        img = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception:
        return img_bytes          # synthetic bytes → passthrough
    w, h = img.size
    if max(w, h) <= long_edge:        # don't upscale
        target_w, target_h = w, h
    elif w >= h:
        target_w, target_h = long_edge, int(h * long_edge / w)
    else:
        target_w, target_h = int(w * long_edge / h), long_edge
    resized = img.resize((target_w, target_h), PILImage.LANCZOS)
    buf = io.BytesIO()
    resized.save(buf, format="JPEG", quality=quality, optimize=True)
    return buf.getvalue()


def generate_derivatives(
    photo:    PhotoRow,
    s3_client,
    source_bucket: str,
) -> dict[str, str]:
    """Download original from `source_bucket`, generate three derivatives,
    upload each to DERIVATIVE_BUCKET, return the per-size presigned GET URLs.
    Idempotent: same photo in → same s3 keys out, PUTs overwrite with same bytes.
    """
    # 1. Download original.
    obj = s3_client.get_object(Bucket=source_bucket, Key=photo.s3_key)
    img_bytes = obj["Body"].read()

    # 2. Generate + upload each derivative.
    out: dict[str, str] = {}
    for size_name, (long_edge, quality) in DERIVATIVE_SIZES.items():
        derivative_bytes = _resize_lanczos(img_bytes, long_edge, quality)
        key = _derivative_key(photo.sha256, size_name)
        s3_client.put_object(
            Bucket=DERIVATIVE_BUCKET,
            Key=key,
            Body=derivative_bytes,
            ContentType="image/jpeg",
        )
        out[size_name] = key
    return out


## Verifying Derivative Generation Against Mocks

We stub S3 with `MagicMock` and inspect the resulting `put_object` calls. The contract: three `put_object` calls per photo, one per size, each with the right key shape and content type, and each body smaller than the original (sanity check the resize actually happened).


In [ ]:
from unittest.mock import MagicMock

mock_s3 = MagicMock()
mock_s3.get_object.return_value = {
    "Body": MagicMock(read=lambda: b"\xff\xd8\xff\xe0" + b"\x00" * 1000)   # fake JPEG header + body
}

photo = PhotoRow(photo_id="p-001", s3_key="photos/aa/abcd.jpg", sha256="a" * 64)

# Without Pillow installed, _resize_lanczos returns the input bytes unchanged.
# With Pillow, the returned bytes would be a real JPEG thumbnail.
urls = generate_derivatives(photo, mock_s3, source_bucket="my-photos")

print(f"put_object called: {mock_s3.put_object.call_count} times  (expected 3)")
print(f"uploaded keys: {list(urls.values())}")
print(f"thumbnails key shape correct: {all(k.startswith('thumbnails/aa/') for k in urls.values())}")

# Verify per-size: the long-edge limit was respected (in the stub case, we cannot
# actually decode the resize, but we can verify the function correctly iterated
# over DERIVATIVE_SIZES).
expected_sizes = set(DERIVATIVE_SIZES.keys())
actual_sizes = set(urls.keys())
print(f"all three sizes generated: {expected_sizes == actual_sizes}")


## Derivative URLs in API Responses

The `GET /photos/` response now includes URLs for each size, not just the original. The client picks per viewport (timeline → `small`; detail → `large`). Adding a `size` hint parameter to the request lets the client ask the API to return **only** the URL it needs (smaller JSON, less serialization work); passing no hint returns all sizes plus the original (`original` only when `?include_original=true`).

:::{.callout-note}
Returning all sizes by default has the benefit that the client always has the URL it needs without a second round-trip. The cost is JSON size: each presigned URL is ~500 bytes, four of them = 2 KB per photo in the response. For a 30-photo page that's 60 KB of JSON, which is well below the cost of any one photo's `small` thumbnail. Acceptable.

:::


In [ ]:
from datetime import datetime
from pydantic import BaseModel


class PhotoURLs(BaseModel):
    """All URLs the client may need for a single photo. None means not generated
    yet (the backfill job in the next section will populate it)."""
    small:    str | None = None
    medium:   str | None = None
    large:    str | None = None
    original: str | None = None


class BrowsePhoto(BaseModel):
    photo_id:  str
    taken_at:  datetime
    width_px:  int = 0           # populated by the pipeline; 0 if unknown
    height_px: int = 0
    urls:      PhotoURLs


def _presign_get(s3_client, bucket: str, key: str, expires_in: int = 3600) -> str:
    return s3_client.generate_presigned_url(
        "get_object",
        Params={"Bucket": bucket, "Key": key},
        ExpiresIn=expires_in,
    )


def build_photo_urls(
    s3_client,
    original_bucket: str,
    photo:           dict,
    derivative_keys: dict[str, str],   # {"small": "thumbnails/...", ...} (may be missing)
    include_original: bool = False,
    presign_expiry_s: int = 3600,
) -> PhotoURLs:
    urls = PhotoURLs()
    for size_name in ("small", "medium", "large"):
        if key := derivative_keys.get(size_name):
            setattr(urls, size_name, _presign_get(s3_client, DERIVATIVE_BUCKET, key, presign_expiry_s))
    if include_original and photo.get("s3_key"):
        urls.original = _presign_get(s3_client, original_bucket, photo["s3_key"], presign_expiry_s)
    return urls


# Demonstrate.
s3 = MagicMock()
s3.generate_presigned_url.side_effect = lambda op, Params, ExpiresIn: (
    f"https://{Params['Bucket']}.s3.example.com/{Params['Key']}?GET&exp={ExpiresIn}"
)
photo_row = {"photo_id": "p-001", "s3_key": "photos/aa/abcd.jpg"}
derivs = {
    "small":  "thumbnails/aa/aaaa/small.jpg",
    "medium": "thumbnails/aa/aaaa/medium.jpg",
    "large":  "thumbnails/aa/aaaa/large.jpg",
}

print("With include_original=False (default, browse use):")
print(build_photo_urls(s3, "my-photos", photo_row, derivs).model_dump_json(indent=2))
print("\nWith include_original=True (detail use):")
print(build_photo_urls(s3, "my-photos", photo_row, derivs, include_original=True).model_dump_json(indent=2))


## Face Crops From the Medium Derivative

The single largest cost in notebook 12's people albums is `generate_face_thumbnail` downloading the original photo so it can crop to a face bounding box. Each face = one full-photo download. A person with 500 photos = 500 full downloads.

The fix is to crop from the **medium** derivative instead of the original. The medium derivative is a 512 × 384 JPEG weighing ~50 KB instead of ~5 MB — a 100× reduction in bytes downloaded. The face bbox is in original-image pixel coordinates, so we must **scale the bbox to the medium's coordinate space** before cropping.

The trade-off: a 256 × 256 face thumbnail cropped from medium (which is 512 px on the long edge) is upscaling, not downscaling. For most faces this is imperceptible because the face itself is rarely the long edge of the photo. For very small faces (background people in a wide-angle shot), the medium may not have enough pixels to crop a clean 256 × 256 — but those faces would also not be useful as person album covers anyway, and the pipeline's clustering pass already filters them.


In [ ]:
@dataclass
class FaceRow:
    face_id:  str
    photo_id: str
    bbox_x:   int
    bbox_y:   int
    bbox_w:   int
    bbox_h:   int


@dataclass
class PhotoDim:
    """Original photo dimensions — needed to rescale bbox from original to medium."""
    width_px:  int
    height_px: int


def _scale_bbox_to_derivative(
    face: FaceRow, photo: PhotoDim, derivative_long_edge: int = 512,
) -> tuple[int, int, int, int]:
    """Rescale face bbox from original image coords to derivative coords."""
    scale = min(derivative_long_edge / photo.width_px, derivative_long_edge / photo.height_px)
    scale = min(scale, 1.0)   # never upscale bbox geometry
    return (
        int(face.bbox_x * scale),
        int(face.bbox_y * scale),
        int(face.bbox_w * scale),
        int(face.bbox_h * scale),
    )


def generate_face_thumbnail_from_derivative(
    face:       FaceRow,
    photo:      PhotoRow,
    photo_dim:  PhotoDim,
    s3_client,
    source_bucket: str = DERIVATIVE_BUCKET,    # read medium, not original
    source_size:   str = "medium",             # which derivative to crop from
    padding_factor: float = 0.20,
    expiry_s: int = 3600,
) -> str:
    """Crop face from the medium derivative instead of the original."""
    # 1. Download derivative (not original). ~50 KB instead of ~5 MB.
    derivative_key = _derivative_key(photo.sha256, source_size)
    obj = s3_client.get_object(Bucket=source_bucket, Key=derivative_key)
    img_bytes = obj["Body"].read()

    if not _PIL_AVAILABLE:
        thumbnail_bytes = img_bytes
    else:
        img = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        # 2. Rescale bbox to derivative coordinate space.
        sx, sy, sw, sh = _scale_bbox_to_derivative(
            face, photo_dim, DERIVATIVE_SIZES[source_size][0],
        )
        pad_x = int(sw * padding_factor)
        pad_y = int(sh * padding_factor)

        # The derivative's actual size: long edge = 512, short edge proportional.
        deriv_w, deriv_h = img.size
        left   = max(0, sx - pad_x)
        top    = max(0, sy - pad_y)
        right  = min(deriv_w, sx + sw + pad_x)
        bottom = min(deriv_h, sy + sh + pad_y)
        crop = img.crop((left, top, right, bottom))
        buf = io.BytesIO()
        crop.save(buf, format="JPEG", quality=85)
        thumbnail_bytes = buf.getvalue()

    # 3. Upload face thumbnail. Hash-stable face key: derived from face_id.
    thumb_key = f"thumbnails/faces/{face.face_id}.jpg"
    s3_client.put_object(
        Bucket=DERIVATIVE_BUCKET, Key=thumb_key, Body=thumbnail_bytes, ContentType="image/jpeg",
    )
    return _presign_get(s3_client, DERIVATIVE_BUCKET, thumb_key, expiry_s)


# Verify.
mock_s3 = MagicMock()
mock_s3.get_object.return_value = {"Body": MagicMock(read=lambda: b"\xff\xd8" + b"\x00" * 5000)}
mock_s3.generate_presigned_url.return_value = "https://s3.example.com/face.jpg?exp=3600"

fr = FaceRow("face-001", "p-001", 120, 80, 100, 100)
pr = PhotoRow("p-001", "photos/aa/abcd.jpg", sha256="a" * 64)
dm = PhotoDim(width_px=1920, height_px=1080)

url = generate_face_thumbnail_from_derivative(fr, pr, dm, mock_s3)
bucket_called = mock_s3.get_object.call_args_list[0].kwargs["Bucket"]
key_called = mock_s3.get_object.call_args_list[0].kwargs["Key"]
print(f"downloaded from bucket: {bucket_called}  (should be {DERIVATIVE_BUCKET})")
print(f"downloaded key:        {key_called}")
print(f"face thumbnail URL:    {url}")
print(f"downloaded medium not original: {bucket_called != 'my-photos'}")


## On-Demand Backfill for Missing Derivatives

Photos indexed before the derivative pipeline existed (the legacy `aws s3 sync` upload from notebook 12) have derivatives missing. We never block browsing waiting for backfill; we do a **just-in-time** generation in a background task.

The contract: `build_photo_urls` returns a `PhotoURLs` with `None` for any size whose derivative key is not yet known. The client treats `None` as a "small" placeholder URL — or, if we want a graceful degradation, requests the **next-largest available** size and the browser downscales. The pipeline (background) sees the row's missing-derivative state, generates them, and broadcasts a "thumbnail ready" event over the WebSocket so the client refreshes just that cell.


In [ ]:
# A backfill dispatcher enrolled by the pipeline at end of per-photo indexing.
async def ensure_derivatives(photo_row: dict, s3_client) -> dict[str, str]:
    """Backfill missing derivatives. Returns the resulting derivative key map."""
    sha = photo_row["sha256"]
    existing = photo_row.get("derivative_keys", {})  # {"small": "...", ...}
    out = dict(existing)
    for size_name in DERIVATIVE_SIZES:
        if size_name not in existing:
            # generate_derivatives (defined above) re-runs for ALL three sizes.
            # In production we'd skip the ones in `existing`; for clarity we
            # call it once and let it overwrite. Idempotent, so safe.
            keys = generate_derivatives(
                PhotoRow(photo_id=photo_row["photo_id"], s3_key=photo_row["s3_key"], sha256=sha),
                s3_client,
                source_bucket="my-photos",
            )
            out.update(keys)
            break
    return out


# Simulate a legacy row with no derivatives, then backfill.
mock_s3 = MagicMock()
mock_s3.get_object.return_value = {"Body": MagicMock(read=lambda: b"\xff\xd8\xff" + b"\x00" * 1000)}
legacy = {"photo_id": "p-legacy", "sha256": "b" * 64, "s3_key": "photos/bb/bbcd.jpg"}
filled = await ensure_derivatives(legacy, mock_s3)
print(f"backfilled")
print(f"keys: {list(filled.keys())}")
print(f"3 sizes present: {set(filled.keys()) == set(DERIVATIVE_SIZES.keys())}")


## S3 Cache-Control and Content-Type on Derivatives

Derivatives are immutable (their key is hash-stable, so a key always points at the same bytes). That makes them ideal candidates for aggressive `Cache-Control` headers — both for CDN edge caching and for the Flet client's HTTP cache. We set `Cache-Control: public, max-age=31536000, immutable` on every derivative `put_object`. The original gets a shorter cache TTL because it could in principle be replaced (e.g. by a re-export), and we want clients to refetch. The presigned URL contains query-string expiry that limits total usability to ~1 hour, but **the bytes at the resolved URL are cached** for the year — the client never even re-issues the GET for that key within the cache window.

:::{.callout-note}
S3 `Cache-Control` headers apply once per `put_object` and cannot be changed without re-uploading. This is why immutable keys are critical: if the same key could map to different bytes (which our hash-stable scheme prevents), a client would cache the old bytes indefinitely and never see the new ones.

:::

```python
# (illustrative — not executed, just the production put_object call shape)
s3_client.put_object(
    Bucket=DERIVATIVE_BUCKET,
    Key=key,
    Body=derivative_bytes,
    ContentType="image/jpeg",
    CacheControl="public, max-age=31536000, immutable",
    Metadata={"derivative-of-sha256": photo.sha256, "size-name": size_name},
)
```

---

## Schema Additions

A new `photo_derivatives` table links each photo's sha256 to its derivative S3 keys, with a generation timestamp so the pipeline can know whether to skip or regenerate.Keeping this map in a side table — not columns on `photos` — lets us add a new size (`xlarge`, say) without an `ALTER TABLE`.

```sql
-- 0019_pht_derivatives.sql
CREATE TABLE photo_derivatives (
    photo_id      TEXT NOT NULL REFERENCES photos(photo_id),
    size_name     TEXT NOT NULL CHECK (size_name IN ('small', 'medium', 'large')),
    s3_key        TEXT NOT NULL,
    generated_at  TIMESTAMPTZ NOT NULL DEFAULT now(),
    PRIMARY KEY (photo_id, size_name)
);

CREATE INDEX ix_derivatives_pending
    ON photos (photo_id)
    WHERE NOT EXISTS (
        SELECT 1 FROM photo_derivatives pd
        WHERE pd.photo_id = photos.photo_id AND pd.size_name = 'small'
    );
-- The partial index above is illustrative; in production a more practical
-- alternative is a `derivatives_state` enum column on photos ('none'|'partial'|'complete')
-- updated by the pipeline as each derivative is generated.
```


## Summary

This notebook introduced the derivative tier — three JPEG derivatives per photo (`small` 256, `medium` 512, `large` 1024) generated at index time, stored in a sibling bucket, and keyed by the photo's sha256 for idempotency. We rebuilt notebook 12's `generate_face_thumbnail` to crop from the medium derivative instead of the original, turning the 5 MB-per-face download into a 50 KB-per-face download. The derivative API response lets the client pick per viewport; missing derivatives are backfilled just-in-time by the pipeline and signaled to the client via the existing WebSocket channel.

**What changed relative to notebook 12.** The `GET /photos/` response shape went from `presigned_url: str` to `urls: PhotoURLs`. The `generate_face_thumbnail` function now downloads from `my-photo-app-thumbnails` instead of the original bucket, and rescales the face bbox before cropping. The pipeline gained a derivative-generation step; the schema gained a `photo_derivatives` table.

**What this enables for the rest of PHT.** The next notebook (PHT:03 — Efficient Browsing) takes the `small` derivative URL as the default browse output, and reduces the presigned-URL generation from "regenerate per request" to "regenerate per hour and cache." PHT:04's search results point at `small` for the result grid. PHT:05's durable pipeline makes derivative generation a checkpointed step. PHT:06's delete cascade extends to the `my-photo-app-thumbnails` bucket. All of these notebooks count on the derivative tier being there.

---



---


■
